# AIPlasma v1 — Tutorial

AIPlasma is a physics-informed machine-learning framework for plasma physics and fusion research, built on PyTorch. This notebook is a guided tour of everything that ships in **v1**, one layer at a time.

| # | Section | What you will learn |
|---|---------|---------------------|
| 0 | Setup | Imports, repository layout, design principles |
| 1 | Data layer | `PhysicalTensor`, data sources, fidelity levels, `FeatureBatch`, `DataPipeline` |
| 2 | Physics equations | `PDERegistry`, built-in equations, registering your own |
| 3 | Models | `BasePINN`, `ResidualPINN`, `MultiFidelityPINN` |
| 4 | Solvers | `AutogradPDESolver`, ODE solvers, the dual-mode solver interface |
| 5 | Training | `PhysicsProblem`, `CompiledProblem`, the three trainers, callbacks |
| 6 | Uncertainty | `BayesianNN` (MC Dropout), `EnsembleModel`, `UncertaintyMetrics` |
| 7 | End to end | Raw data source to trained multi-fidelity model |
| 8 | Limitations | What is stubbed in v1 |

**How to run.** Start Jupyter from the repository (or any sub-folder of it). Every cell runs on CPU. The training runs are deliberately short: they demonstrate the API, they are not tuned for accuracy.

## 0. Setup and design principles

### Layers

```
data/       PhysicalTensor -> DataSource -> FidelityAssigner -> DataTransformer -> FeatureBatch
solvers/    PDERegistry (physics_constraints), AutogradPDESolver, Euler / RK4 / Adaptive ODE solvers
models/     BasePINN, ResidualPINN, MultiFidelityPINN, BayesianNN, EnsembleModel
core/       PhysicsProblem, PhysicsModel, PhysicsSolver, BaseTrainer, CompiledProblem, domain problems
trainers/   PINNTrainer, BayesianTrainer, MultiFidelityTrainer, callbacks
```

### Principles you will see repeatedly

* **Subclassing is the primary API** (ADR-001). You extend `PhysicsProblem`, `PhysicsEquation`, `Callback`, and so on.
* **Solvers are dual-mode** (ADR-003). `solve()` post-processes a prediction without autograd; `solve_with_grad()` runs the model with the autograd graph intact, so losses can be back-propagated.
* **The training loop is sealed** (ADR-004). `BaseTrainer.fit()` owns the loop; subclasses fill in abstract steps. `fit()` takes a `list[FeatureBatch]`, not a `DataLoader`.
* **Everything flows through `FeatureBatch`.** Data sources, models, solvers and trainers all speak this one container.

In [ ]:
import csv
import math
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Locate the repository root: the folder that contains core/ and models/.
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "core").is_dir() and (p / "models").is_dir()
)
sys.path.insert(0, str(ROOT))
# data_parser is imported as a top-level module in the test suite, so data/ is on the path too.
if (ROOT / "data").is_dir():
    sys.path.insert(0, str(ROOT / "data"))

torch.manual_seed(0)
print("Repository root:", ROOT)
print("PyTorch        :", torch.__version__)

In [ ]:
# --- Data layer ---
from data.sources.physical_tensor import PhysicalTensor, UnitSystem, Domain, CoordinateSystem
from data.preprocessing.feature_pipeline import FeatureBatch
from data.preprocessing.data_transformer import DataTransformer, FidelityAssigner, FeatureBatchDataset
from data_parser import DataSourceConfig, SyntheticDataSource, CSVDataSource, HDF5DataSource
from data.data_pipeline import DataPipeline

# --- Physics equations and solvers ---
from solvers.physics_constraints import REGISTRY, PDERegistry, PhysicsEquation
from solvers.pde_solver import AutogradPDESolver
from solvers.ode_solver import EulerODESolver, RK4ODESolver, AdaptiveODESolver

# --- Core interfaces ---
from core.base_model import ModelOutput
from core.base_problem import PhysicsProblem, CompiledProblem
from core.base_solver import SolverOutput, SolverInfo
from core.base_trainer import TrainerConfig, TrainingHistory

# --- Models ---
from models.pinn.base_pinn import BasePINN
from models.pinn.residual_pinn import ResidualPINN
from models.pinn.multi_fidelity_pinn import MultiFidelityPINN
from models.bayesian.bayesian_nn import BayesianNN
from models.bayesian.uncertenty import UncertaintyMetrics
from models.ensemble.ensemble_model import EnsembleModel

# --- Trainers and callbacks ---
from trainers.pinn_trainer import PINNTrainer
from trainers.bayesian_trainer import BayesianTrainer
from trainers.callbacks.mf_trainer import MultiFidelityConfig, MultiFidelityTrainer
from trainers.callbacks.base_callback import Callback
from trainers.callbacks.check_pointing import CheckPointing
from trainers.callbacks.early_stopping import EarlyStopping

print("All AIPlasma imports succeeded.")

## 1. Data layer

Plasma data comes from very different places: analytic solutions, CSV exports, HDF5 files from gyrokinetic codes. The data layer turns all of them into one representation and then into training-ready batches.

```
DataSource --load()--> PhysicalTensor --DataTransformer--> FeatureBatch --> trainer
     (many kinds)      (values + coords + units + domain)   (masks, weights, fidelity)
```

### 1.1 `SyntheticDataSource` and `PhysicalTensor`

A `DataSourceConfig` describes *what the data means*: the coordinate system, the physical units and the domain. Our running example is the 1-D heat equation $u_t = \alpha\, u_{xx}$ on $x, t \in [0, 1]$ with the exact solution $u(x,t) = \sin(\pi x)\, e^{-\alpha \pi^2 t}$.

In [ ]:
def heat_solution(x, t, alpha=0.01):
    """Exact solution of u_t = alpha * u_xx with u(0, t) = u(1, t) = 0 and u(x, 0) = sin(pi x)."""
    return torch.sin(math.pi * x) * torch.exp(torch.tensor(-alpha * math.pi ** 2) * t)


def make_config() -> DataSourceConfig:
    """A fresh config each time, because fields such as max_points are mutable."""
    return DataSourceConfig(
        coord_system=CoordinateSystem.CARTESIAN,
        units=UnitSystem(spatial="m", temporal="s", field="normalized"),
        domain=Domain(x_range=(0.0, 1.0), t_range=(0.0, 1.0)),
    )


source = SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=300)
print("validate():", source.validate())

tensor = source.load()
print("PhysicalTensor:", isinstance(tensor, PhysicalTensor))
print("n_points    :", tensor.n_points())
print("n_fields    :", tensor.n_fields())
print("coordinates :", tuple(tensor.coordinates.shape))
print("values      :", tuple(tensor.values.shape))
print("coord system:", tensor.coord_system)
print("metadata    :", tensor.metadata)
print("all points inside the domain:", tensor.domain.contains(tensor.coordinates).all().item())

In [ ]:
plt.figure(figsize=(5, 4))
sc = plt.scatter(
    tensor.coordinates[:, 0].numpy(), tensor.coordinates[:, 1].numpy(),
    c=tensor.values[:, 0].numpy(), s=12,
)
plt.colorbar(sc, label="u(x, t)")
plt.xlabel("x")
plt.ylabel("t")
plt.title("SyntheticDataSource: 300 random points")
plt.tight_layout()
plt.show()

### 1.2 Noise, point limits and validation

Sources can add Gaussian noise, cap the number of points through `config.max_points`, and report whether they are usable through `validate()`.

In [ ]:
# Gaussian noise on the field values
noisy = SyntheticDataSource(
    solution_fn=heat_solution, config=make_config(), n_points=100, noise_std=0.05
).load()
print("noise_std recorded in metadata:", noisy.metadata.get("noise_std"))

# Cap the number of points
limited_cfg = make_config()
limited_cfg.max_points = 50
limited = SyntheticDataSource(solution_fn=heat_solution, config=limited_cfg, n_points=200).load()
print("points with max_points=50    :", limited.n_points())

# A config without a domain is invalid
bad_cfg = DataSourceConfig(
    coord_system=CoordinateSystem.CARTESIAN,
    units=UnitSystem(spatial="m", temporal="s", field="normalized"),
    domain=None,
)
print("validate() without a domain  :", SyntheticDataSource(solution_fn=heat_solution, config=bad_cfg).validate())

### 1.3 Streaming

Large datasets should not have to fit in memory at once. `stream(chunk_size)` yields the data as a sequence of `PhysicalTensor` chunks.

In [ ]:
chunks = list(
    SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=100).stream(chunk_size=30)
)
print("chunks     :", len(chunks))
print("chunk sizes:", [c.n_points() for c in chunks])

### 1.4 `CSVDataSource`

A CSV source needs the path, the names of the field columns, and a config. Here we write a temporary CSV with columns `x, t, u`, then load it.

In [ ]:
def write_heat_csv(path: Path, n_rows: int, alpha: float = 0.01) -> None:
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["x", "t", "u"])
        for _ in range(n_rows):
            x, t = float(torch.rand(1)), float(torch.rand(1))
            writer.writerow([x, t, math.sin(math.pi * x) * math.exp(-alpha * math.pi ** 2 * t)])


with tempfile.TemporaryDirectory() as tmp:
    csv_path = Path(tmp) / "heat.csv"
    write_heat_csv(csv_path, n_rows=120)

    csv_source = CSVDataSource(path=str(csv_path), field_columns=["u"], config=make_config())
    print("validate():", csv_source.validate())
    csv_tensor = csv_source.load()
    print("n_points  :", csv_tensor.n_points())
    print("metadata  :", csv_tensor.metadata)

# A missing file is reported by validate() and rejected by load()
missing = CSVDataSource(path="/nonexistent/data.csv", field_columns=["u"], config=make_config())
print("missing file, validate():", missing.validate())
try:
    missing.load()
except ValueError:
    print("missing file, load() raised ValueError")

### 1.5 `HDF5DataSource` (stub in v1)

`HDF5DataSource` reserves the interface for output of codes such as GENE, GS2 and CGYRO. The expected file layout is `/coordinates` with shape `(N, D)` and `/fields` with shape `(N, F)`. Loading is not implemented yet in v1, but the class can be constructed and `validate()` correctly reports a missing file.

In [ ]:
h5_source = HDF5DataSource(path="/nonexistent/data.h5", config=make_config())
print("validate() for a missing file:", h5_source.validate())

### 1.6 Fidelity levels

Real projects mix data of different quality: a coarse fast simulation, a finer one, an experiment. `FidelityAssigner` scores each source (from properties such as resolution) and ranks them into integer **fidelity levels**; a higher level means higher fidelity. The assignment is deterministic.

In [ ]:
coarse = SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=50).load()
medium = SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=150).load()
fine   = SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=400).load()

assigner = FidelityAssigner()
print("one source    :", assigner.assign([coarse]))
print("two sources   :", assigner.assign([coarse, fine]))
print("three sources :", assigner.assign([coarse, medium, fine]))
print("deterministic :", assigner.assign([coarse, medium, fine]) == assigner.assign([coarse, medium, fine]))

try:
    assigner.assign([])
except ValueError:
    print("empty list    : ValueError")

# Sources in incompatible coordinate systems cannot be ranked against each other
toroidal = SyntheticDataSource(
    solution_fn=heat_solution,
    config=DataSourceConfig(
        coord_system=CoordinateSystem.TOROIDAL,
        units=UnitSystem(spatial="m", temporal="s", field="normalized"),
        domain=Domain(x_range=(0.0, 1.0), t_range=(0.0, 1.0)),
    ),
    n_points=50,
).load()
print("cartesian + toroidal compatible:", assigner.validate([coarse, toroidal]))

### 1.7 `DataTransformer` and `FeatureBatch`

`DataTransformer.transform()` converts a list of `PhysicalTensor` objects into a list of `FeatureBatch` objects, one per source. Each batch carries:

| Field | Meaning |
|-------|---------|
| `coords` | Input coordinates, shape `(N, D)` |
| `fields` | Target field values, shape `(N, F)` |
| `boundary_mask` / `collocation_mask` | Boolean masks that split the points into boundary and interior; they never overlap |
| `physics_params` | Equation parameters (for example `{"alpha": 0.01}`) |
| `fidelity_level` / `fidelity_weight` | Integer level and the loss weight of this source; weights sum to 1 |

In [ ]:
transformer = DataTransformer()
batches = transformer.transform([coarse, medium, fine])

for b in batches:
    overlap = (b.boundary_mask & b.collocation_mask).any().item()
    print(
        f"level={b.fidelity_level}  weight={b.fidelity_weight:.3f}  "
        f"coords={tuple(b.coords.shape)}  boundary={int(b.boundary_mask.sum())}  "
        f"interior={int(b.collocation_mask.sum())}  masks overlap={overlap}  "
        f"fields in [{b.fields.min().item():.3f}, {b.fields.max().item():.3f}]"
    )
print("weights sum to:", round(sum(b.fidelity_weight for b in batches), 6))

### 1.8 `FeatureBatchDataset` and `DataPipeline`

`DataPipeline` chains everything above: sources go in, and a train/validation pair of `FeatureBatchDataset` comes out. Note that the split is made **between sources** (each source becomes one batch), so the two datasets together always contain as many batches as there are sources.

In [ ]:
pipeline = DataPipeline(sources=[
    SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=50),
    SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=150),
    SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=400),
])
train_ds, val_ds = pipeline.build()

print("train batches:", len(train_ds), "| validation batches:", len(val_ds))
print("fidelity levels present:", sorted({ds[i].fidelity_level for ds in (train_ds, val_ds) for i in range(len(ds))}))
print()
print(pipeline.describe())

A `FeatureBatchDataset` can also be built by hand from a list of batches and split with `split(val_ratio=...)`, which is how we will organise the training data in section 5.

In [ ]:
dataset = FeatureBatchDataset(batches)          # the three batches from section 1.7
train_part, val_part = dataset.split(val_ratio=0.33)
print("dataset size:", len(dataset), "| first item is a FeatureBatch:", isinstance(dataset[0], FeatureBatch))
print("split ->", len(train_part), "train /", len(val_part), "validation")

## 2. Physics equations

Equations live in **`PDERegistry`**, a singleton that maps a name to a `PhysicsEquation` object. Solvers and problems look equations up by name, so adding a new equation never requires touching framework code.

Every `PhysicsEquation` provides `name()`, `expected_params()`, `description()`, `validate_params()` and `residual(coords, pred, params)`. The residual is the quantity a PINN drives to zero: it is `0` where the prediction satisfies the PDE.

In [ ]:
print("Registered equations:", REGISTRY.list_all())
print()
for name in ["heat_equation_1d", "drift_diffusion_1d", "hasegawa_wakatani"]:
    eq = REGISTRY.get(name)
    print(f"{eq.name():20s} params={eq.expected_params()}")
    print(f"{'':20s} {eq.description()}")

# Lookups of unknown names fail loudly
try:
    REGISTRY.get("nonexistent_equation")
except KeyError:
    print("\nREGISTRY.get('nonexistent_equation') -> KeyError")

### 2.1 Computing a residual

The exact heat-equation solution should give a residual that is numerically close to zero. Because the residual is built with autograd, the coordinates must require gradients.

In [ ]:
heat = REGISTRY.get("heat_equation_1d")
print("validate_params:", heat.validate_params({"alpha": 0.01}))
try:
    heat.validate_params({})
except ValueError:
    print("validate_params({}) -> ValueError (missing 'alpha')")

coords = torch.rand(100, 2, requires_grad=True)          # columns: x, t
pred = heat_solution(coords[:, 0], coords[:, 1]).unsqueeze(1)

residual = heat.residual(coords, pred, {"alpha": 0.01})
print("residual shape           :", tuple(residual.shape))
print("mean |residual| (exact u):", residual.abs().mean().item())

The residual is differentiable, so it can be used directly inside a loss. Below, a tiny untrained network gives a large residual, and back-propagating through it produces non-zero gradients.

In [ ]:
net = torch.nn.Linear(2, 1)
coords = torch.rand(50, 2, requires_grad=True)
res = heat.residual(coords, net(coords), {"alpha": 0.01})
res.pow(2).mean().backward()
print("gradient norm of the linear layer:", net.weight.grad.norm().item())

### 2.2 Registering your own equation

Subclass `PhysicsEquation`, implement the required methods, and register it. Here is an exponential decay $u_t + k\,u = 0$.

In [ ]:
class ExponentialDecay(PhysicsEquation):
    """u_t + k * u = 0"""

    def name(self):
        return "exponential_decay"

    def expected_params(self):
        return ["k"]

    def description(self):
        return "Exponential decay: u_t + k u = 0"

    def residual(self, coords, pred, params):
        u_t = torch.autograd.grad(pred.sum(), coords, create_graph=True)[0][:, 1:2]
        return u_t + params["k"] * pred


REGISTRY.register(ExponentialDecay())
try:
    print("registered:", "exponential_decay" in REGISTRY)

    coords = torch.rand(40, 2, requires_grad=True)
    k = 0.5
    exact = torch.exp(-k * coords[:, 1:2])                 # u = exp(-k t) solves the equation
    res = REGISTRY.get("exponential_decay").residual(coords, exact, {"k": k})
    print("mean |residual| for the exact solution:", res.abs().mean().item())

    # Registering a duplicate name is rejected
    try:
        REGISTRY.register(ExponentialDecay())
    except ValueError:
        print("duplicate registration -> ValueError")
finally:
    REGISTRY.unregister("exponential_decay")
print("after unregister:", "exponential_decay" in REGISTRY)

### 2.3 `HasegawaWakatani` (stub in v1)

The Hasegawa–Wakatani equations are registered so that the name, parameters (`D`, `C`, `nu`) and description exist, but `residual()` raises `NotImplementedError` until the plasma operators (Poisson bracket) are wired in.

In [ ]:
hw = REGISTRY.get("hasegawa_wakatani")
print("expected_params:", hw.expected_params())
try:
    hw.residual(None, None, {"D": 0.1, "C": 1.0, "nu": 0.01})
except NotImplementedError:
    print("residual() -> NotImplementedError (stub)")

## 3. Models

All models inherit from `PhysicsModel`, which is both an `nn.Module` and an abstract base class. That gives you `parameters()`, `state_dict()` and `train()` / `eval()` for free. A model's `forward(batch)` takes a `FeatureBatch` and returns a `ModelOutput` with two fields:

* `pred`: predictions, shape `(N, output_dim)`
* `uncertainty`: `None` for deterministic models, a tensor for probabilistic ones

For the rest of the tutorial we use a small helper that builds a `FeatureBatch` directly. This makes every field visible and keeps coordinates in physical units, which is convenient for plotting.

In [ ]:
def make_batch(n: int = 100, fidelity_level: int = 0, alpha: float = 0.01) -> FeatureBatch:
    """Random points in [0, 1]^2 labelled with the exact heat solution."""
    x = torch.rand(n)
    t = torch.rand(n)
    boundary = (x < 0.05) | (x > 0.95)
    return FeatureBatch(
        coords=torch.stack([x, t], dim=1),
        fields=heat_solution(x, t, alpha).unsqueeze(1),
        boundary_mask=boundary,
        collocation_mask=~boundary,
        physics_params={"alpha": alpha, "D": 0.1, "v": 0.5},
        fidelity_level=fidelity_level,
        fidelity_weight=1.0,
    )


def with_grad(batch: FeatureBatch) -> FeatureBatch:
    """Same batch, but with coordinates that require gradients (needed for derivatives)."""
    return FeatureBatch(
        coords=batch.coords.clone().requires_grad_(True),
        fields=batch.fields,
        boundary_mask=batch.boundary_mask,
        collocation_mask=batch.collocation_mask,
        physics_params=batch.physics_params,
        fidelity_level=batch.fidelity_level,
        fidelity_weight=batch.fidelity_weight,
    )


batch = make_batch()
print("coords:", tuple(batch.coords.shape), "| fields:", tuple(batch.fields.shape))
print("boundary points:", int(batch.boundary_mask.sum()), "| collocation points:", int(batch.collocation_mask.sum()))

### 3.1 `BasePINN`

A fully connected network with a configurable depth, width and activation (`"tanh"`, `"relu"` or `"silu"`).

In [ ]:
pinn = BasePINN(input_dim=2, output_dim=1, hidden_layers=3, hidden_size=32, activation="tanh")

out = pinn.forward(batch)
print("returns ModelOutput:", isinstance(out, ModelOutput))
print("pred shape         :", tuple(out.pred.shape))
print("uncertainty        :", out.uncertainty)
print()
print(pinn.summary())

# predict() is forward() without gradient tracking, for inference
pinn.eval()
print("\npredict() pred shape:", tuple(pinn.predict(batch).pred.shape))

# Unknown activations are rejected
try:
    BasePINN(input_dim=2, output_dim=1, activation="sigmoid")
except ValueError:
    print("activation='sigmoid' -> ValueError")

**Saving and loading.** `save(path)` and `load(path)` round-trip the weights.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = str(Path(tmp) / "pinn.pt")
    pinn.save(path)

    restored = BasePINN(input_dim=2, output_dim=1, hidden_layers=3, hidden_size=32, activation="tanh")
    restored.load(path)
    restored.eval()
    print("predictions preserved:", torch.allclose(pinn.forward(batch).pred, restored.forward(batch).pred, atol=1e-6))

**Derivatives.** `compute_gradients(pred, coords)` returns the derivatives a PDE residual needs: `du_dx`, `du_dt` and `d2u_dx2`. The coordinates in the batch must require gradients, which `with_grad()` handles. The derivatives stay differentiable, so they can be used in a loss.

In [ ]:
gbatch = with_grad(batch)
pred = pinn.forward(gbatch).pred
grads = pinn.compute_gradients(pred, gbatch.coords)

for name, value in grads.items():
    print(f"{name:8s} shape={tuple(value.shape)}")

loss = grads["du_dx"].pow(2).mean() + grads["du_dt"].pow(2).mean()
loss.backward()
print("back-propagation through the derivatives works")

### 3.2 `ResidualPINN`

Same interface as `BasePINN`, with residual (skip) connections between hidden layers. It inherits `compute_gradients()`.

In [ ]:
rpinn = ResidualPINN(input_dim=2, output_dim=1, hidden_layers=4, hidden_size=32)
base4 = BasePINN(input_dim=2, output_dim=1, hidden_layers=4, hidden_size=32)

count = lambda m: sum(p.numel() for p in m.parameters())
print("ResidualPINN parameters:", count(rpinn))
print("BasePINN parameters    :", count(base4))
print("forward pred shape     :", tuple(rpinn.forward(batch).pred.shape))

### 3.3 `MultiFidelityPINN`

`MultiFidelityPINN` extends `ResidualPINN` (ADR-006) with a learned **fidelity embedding**: the batch's `fidelity_level` selects an embedding vector that is fed to the network next to the coordinates. One network can therefore learn from coarse and fine data at the same time and know which is which. `compute_gradients()` differentiates with respect to the original coordinates only, not the embedding.

In [ ]:
mf = MultiFidelityPINN(
    input_dim=2, output_dim=1,
    n_fidelity_levels=3, fidelity_embed_dim=8,
    hidden_layers=2, hidden_size=16,
)

base = make_batch(fidelity_level=0)
same_points_level2 = FeatureBatch(
    coords=base.coords, fields=base.fields,
    boundary_mask=base.boundary_mask, collocation_mask=base.collocation_mask,
    physics_params=base.physics_params, fidelity_level=2, fidelity_weight=1.0,
)

p0 = mf.forward(base).pred
p2 = mf.forward(same_points_level2).pred
print("same coordinates, different fidelity level -> different predictions:", not torch.allclose(p0, p2, atol=1e-6))

# A level outside 0 .. n_fidelity_levels-1 cannot be embedded
try:
    mf.forward(make_batch(fidelity_level=5))
except (IndexError, RuntimeError):
    print("fidelity_level=5 with n_fidelity_levels=3 -> error")

print()
print(mf.summary())

## 4. Solvers

A `PhysicsSolver` turns a model's output into physical quantities and residuals. Every solver implements the same **dual-mode** interface (ADR-003):

| Method | Autograd | Use |
|--------|----------|-----|
| `solve(output, batch)` | off | Post-processing and monitoring, where you already have a prediction |
| `solve_with_grad(model, batch)` | on | Training, where the model is run inside the solver and the residuals must be differentiable |

Both return a `SolverOutput` with `quantities` (named tensors), `residuals` (named tensors) and `solver_info` (a `SolverInfo` with the solver type, convergence flag, iteration count and wall time).

### 4.1 `AutogradPDESolver`

The primary PDE solver (ADR-010). It differentiates the model output with autograd and evaluates the registered equation. Without an equation it still returns derivatives and the boundary residual, and the PDE residual is zero.

In [ ]:
model = BasePINN(input_dim=2, output_dim=1, hidden_layers=2, hidden_size=16)
batch = make_batch(80)

# ---- solve(): no autograd ----
pde_solver = AutogradPDESolver(equation=REGISTRY.get("heat_equation_1d"))
model.eval()
with torch.no_grad():
    output = model.forward(batch)

result = pde_solver.solve(output, batch)
print("solve() quantities:", sorted(result.quantities))
print("solve() residuals :", result.residuals)
print("solver_type       :", result.solver_info.solver_type)

# ---- solve_with_grad(): autograd on ----
result = pde_solver.solve_with_grad(model, batch)
print("\nsolve_with_grad() quantities:", sorted(result.quantities))
print("solve_with_grad() residuals :", sorted(result.residuals))
print("pde residual shape:", tuple(result.residuals["pde"].shape))
print("bc  residual shape:", tuple(result.residuals["bc"].shape), "(one row per boundary point)")
print("converged:", result.solver_info.converged)

The residuals plug straight into a loss. The loop below minimises the PDE residual plus the boundary residual for the small untrained network above, using nothing but the solver and an optimiser. This is the essence of what `PINNTrainer` automates.

In [ ]:
model = BasePINN(input_dim=2, output_dim=1, hidden_layers=3, hidden_size=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch = make_batch(200)

losses = []
for step in range(300):
    optimizer.zero_grad()
    res = pde_solver.solve_with_grad(model, batch)
    loss = res.residuals["pde"].pow(2).mean() + res.residuals["bc"].pow(2).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(5, 3.5))
plt.semilogy(losses)
plt.xlabel("step")
plt.ylabel("PDE + BC residual loss")
plt.title("Minimising residuals with AutogradPDESolver")
plt.tight_layout()
plt.show()

The solver can also compute derivatives on its own through `compute_derivatives(pred, coords)`.

In [ ]:
gbatch = with_grad(make_batch(80))
pred = model.forward(gbatch).pred
derivs = pde_solver.compute_derivatives(pred, gbatch.coords)
print({k: tuple(v.shape) for k, v in derivs.items()})

### 4.2 ODE solvers

Three time-stepping solvers share the interface and add a `step(model, batch, dt)` method:

| Solver | Model evaluations per step | Notes |
|--------|---------------------------|-------|
| `EulerODESolver` | 1 | First order, cheapest |
| `RK4ODESolver` | 4 | Classical fourth-order Runge–Kutta; exposes the stages `k1` ... `k4` |
| `AdaptiveODESolver` | RK45 pair | Chooses the step size from a tolerance (ADR-008); `step()` returns `(SolverOutput, new_dt)` |

In [ ]:
model = BasePINN(input_dim=2, output_dim=1, hidden_layers=2, hidden_size=16)
batch = make_batch(50)

euler, rk4 = EulerODESolver(), RK4ODESolver()

e = euler.step(model, batch, dt=0.1)
r = rk4.step(model, batch, dt=0.1)
print("Euler quantities:", sorted(e.quantities), "| evaluations:", e.solver_info.iterations)
print("RK4   quantities:", sorted(r.quantities), "| evaluations:", r.solver_info.iterations)

gap = (e.quantities["u_next"] - r.quantities["u_next"]).abs().max().item()
print(f"max |u_next(Euler) - u_next(RK4)| at dt=0.1: {gap:.3e}")

# Both are also usable through the dual-mode interface
print("solve_with_grad() keys:", sorted(euler.solve_with_grad(model, batch).quantities))

In [ ]:
adaptive = AdaptiveODESolver(dt_min=1e-6, dt_max=1.0)

out, new_dt = adaptive.step(model, batch, dt=0.1, tol=1e-3)
print("solver_type :", out.solver_info.solver_type)
print("error_norm  :", out.quantities["error_norm"])
print("suggested dt:", new_dt)

_, dt_tight = adaptive.step(model, batch, dt=0.1, tol=1e-6)
_, dt_loose = adaptive.step(model, batch, dt=0.1, tol=1e-1)
print(f"tolerance 1e-6 -> dt {dt_tight:.4g} | tolerance 1e-1 -> dt {dt_loose:.4g}")

### 4.3 Timing

Every solver has timed variants that fill `solver_info.wall_time`.

In [ ]:
model.eval()
with torch.no_grad():
    output = model.forward(batch)

timed = euler.timed_solve(output, batch)
print(f"timed_solve()           wall_time: {timed.solver_info.wall_time * 1e3:.3f} ms")

timed_grad = euler.timed_solve_with_grad(model, batch)
print(f"timed_solve_with_grad() wall_time: {timed_grad.solver_info.wall_time * 1e3:.3f} ms")
print("both return SolverOutput:", isinstance(timed, SolverOutput) and isinstance(timed_grad, SolverOutput))

# SolverInfo can also be built by hand; unset fields fall back to defaults
info = SolverInfo(solver_type="custom", converged=True, iterations=4, wall_time=0.123, extra={"note": "demo"})
minimal = SolverInfo(solver_type="minimal")
print("defaults:", minimal.converged, minimal.iterations, minimal.wall_time, minimal.extra)

## 5. Training

### 5.1 `PhysicsProblem` and `CompiledProblem`

A `PhysicsProblem` defines *what is being learned*: the PDE residual, the boundary conditions and the loss weights (`w_data`, `w_pde`, `w_bc`). You subclass it (ADR-001). Its `compile(model, data_source)` method binds a problem to a model and data into a `CompiledProblem`.

The problem below is intentionally minimal: its PDE residual is zero and `w_pde = 0`, so the demo runs quickly and deterministically by fitting data and boundary conditions. The physics residuals themselves were demonstrated in sections 2 and 4.

In [ ]:
class HeatProblem(PhysicsProblem):
    """Data fit + boundary conditions for the 1-D heat equation."""

    w_data, w_pde, w_bc = 1.0, 0.0, 1.0

    def pde_residual(self, batch, pred):
        return torch.zeros_like(pred)

    def boundary_conditions(self, batch, pred):
        return pred[batch.boundary_mask] - batch.fields[batch.boundary_mask]

    def fidelity_weights(self):
        return [0.5, 0.5]


problem = HeatProblem()

compiled = problem.compile(
    model=BasePINN(input_dim=2, output_dim=1),
    data_source=SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=50),
)
print("CompiledProblem:", isinstance(compiled, CompiledProblem))
print("data_source is a list:", isinstance(compiled.data_source, list))
print()
print(compiled.describe())

### 5.2 `PINNTrainer`

All trainers share the same shape: build a `TrainerConfig`, construct the trainer with a model, a problem and a solver, and call `fit(train_batches, val_batches)`. It returns a `TrainingHistory`.

In [ ]:
solver = AutogradPDESolver()

train_batches = [make_batch(300, fidelity_level=0), make_batch(300, fidelity_level=1)]
val_batches = [make_batch(200, fidelity_level=0)]

config = TrainerConfig(max_epochs=300, val_frequency=10, log_frequency=50)
pinn = BasePINN(input_dim=2, output_dim=1, hidden_layers=3, hidden_size=32)

trainer = PINNTrainer(model=pinn, problem=problem, solver=solver, config=config)
print("device:", trainer.device)

history = trainer.fit(train_batches=train_batches, val_batches=val_batches)
print("returns TrainingHistory:", isinstance(history, TrainingHistory))
print("epochs recorded        :", len(history.epochs))

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.semilogy(history.train_loss)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("PINNTrainer")
plt.tight_layout()
plt.show()

### 5.3 `BayesianTrainer`

Trains a `BayesianNN` (MC Dropout, ADR-007). It rejects any other model type, and it records uncertainty during training. In v1 it uses the standard loss without an explicit KL term (ADR-011).

In [ ]:
bayes_model = BayesianNN(
    input_dim=2, output_dim=1, hidden_size=32, hidden_layers=3, dropout_rate=0.1, n_samples=30
)
b_trainer = BayesianTrainer(model=bayes_model, problem=problem, solver=solver, config=config)
b_history = b_trainer.fit(train_batches=train_batches, val_batches=val_batches)

print("returns TrainingHistory   :", isinstance(b_history, TrainingHistory))
print("uncertainty history length:", len(b_trainer.uncertainty_history()))

try:
    BayesianTrainer(model=pinn, problem=problem, solver=solver, config=config)
except ValueError:
    print("BayesianTrainer(model=BasePINN) -> ValueError")

### 5.4 `MultiFidelityTrainer`

Pairs with `MultiFidelityPINN`. It weights each batch by its fidelity level and adds a `MultiFidelityConfig`. With `curriculum=True` the trainer activates fidelity levels progressively (ADR-012); with `curriculum=False`, as here, all levels are active from the first epoch.

In [ ]:
mf_model = MultiFidelityPINN(
    input_dim=2, output_dim=1, n_fidelity_levels=2, hidden_layers=3, hidden_size=32
)
mf_config = MultiFidelityConfig(curriculum=False, n_fidelity_levels=2)

mf_trainer = MultiFidelityTrainer(
    model=mf_model, problem=problem, solver=solver, config=config, mf_config=mf_config
)
print("active fidelity levels:", mf_trainer.active_levels())

mf_history = mf_trainer.fit(train_batches=train_batches, val_batches=val_batches)
print("returns TrainingHistory:", isinstance(mf_history, TrainingHistory))

### 5.5 Callbacks

Callbacks hook into the sealed training loop without modifying it. Subclass `Callback` and override any of four hooks:

| Hook | Called |
|------|--------|
| `on_train_start(trainer)` | once, before the first epoch |
| `on_epoch_start(trainer, epoch)` | before every epoch |
| `on_epoch_end(trainer, epoch, history)` | after every epoch |
| `on_train_end(trainer, history)` | once, after the last epoch |

v1 ships `EarlyStopping` (stop when the monitored loss stops improving by at least `min_delta` for `patience` epochs) and `CheckPointing` (save model state to `save_dir`). `PhysicsMonitor` computes physics metrics independently through the solver (ADR-011).

In [ ]:
early_stopping = EarlyStopping(patience=3, min_delta=1e-4)
checkpointing = CheckPointing(save_dir=str(Path(tempfile.gettempdir()) / "aiplasma_checkpoints") + "/")
print("EarlyStopping patience:", early_stopping.patience)
print("CheckPointing save_dir:", checkpointing.save_dir)


class EpochPrinter(Callback):
    """A custom callback: print the epoch number every `every` epochs."""

    def __init__(self, every: int = 100):
        super().__init__()
        self.every = every

    def on_epoch_end(self, trainer, epoch, history):
        if epoch % self.every == 0:
            print(f"finished epoch {epoch}")


# The base-class hooks are no-ops, so they are safe to call directly
cb = Callback()
cb.on_train_start(trainer=None)
cb.on_epoch_start(trainer=None, epoch=1)
cb.on_epoch_end(trainer=None, epoch=1, history=None)
cb.on_train_end(trainer=None, history=None)
print("base Callback hooks run without error")

## 6. Uncertainty

Two mechanisms in v1 estimate how much to trust a prediction, and both report through `ModelOutput.uncertainty`.

### 6.1 `BayesianNN` (MC Dropout)

Dropout is kept **active at inference time**, and several stochastic forward passes are averaged: the mean is the prediction and the spread is the uncertainty (ADR-007). Dropout layers sit after each hidden layer (one per hidden layer).

* In `train()` mode `forward()` returns one stochastic pass and `uncertainty is None`.
* In `eval()` mode `forward()` runs `n_samples` passes and fills `uncertainty`.
* `sample_predictions(batch)` returns `(mean, std)` directly.

In [ ]:
demo = BayesianNN(input_dim=2, output_dim=1, hidden_size=16, n_samples=10)
b = make_batch(100)

demo.train()
out_train = demo.forward(b)
print("train mode: uncertainty is", out_train.uncertainty)
print("two training passes differ (dropout active):", not torch.allclose(demo.forward(b).pred, demo.forward(b).pred))

demo.eval()
out_eval = demo.forward(b)
print("eval mode : uncertainty shape", tuple(out_eval.uncertainty.shape), "| non-negative:", bool((out_eval.uncertainty >= 0).all()))

mean, std = demo.sample_predictions(b)
print("sample_predictions:", tuple(mean.shape), tuple(std.shape))
print("uncertainty() convenience:", tuple(demo.uncertainty(b).shape))

# Invalid hyper-parameters are rejected
for kwargs in ({"dropout_rate": 0.0}, {"dropout_rate": 1.0}, {"activation": "sigmoid"}):
    try:
        BayesianNN(input_dim=2, output_dim=1, **kwargs)
    except ValueError:
        print(f"BayesianNN({kwargs}) -> ValueError")

Now the model trained in section 5.3, evaluated along the line $t = 0.5$ against the exact solution. The band is the mean plus or minus two standard deviations of the MC Dropout samples.

In [ ]:
x_line = torch.linspace(0.0, 1.0, 200)
t_line = torch.full_like(x_line, 0.5)
line_batch = FeatureBatch(
    coords=torch.stack([x_line, t_line], dim=1),
    fields=heat_solution(x_line, t_line).unsqueeze(1),
    boundary_mask=torch.zeros(200, dtype=torch.bool),
    collocation_mask=torch.ones(200, dtype=torch.bool),
    physics_params={"alpha": 0.01},
    fidelity_level=0,
    fidelity_weight=1.0,
)

with torch.no_grad():
    mean, std = bayes_model.sample_predictions(line_batch)

x_np = x_line.numpy()
mean_np, std_np = mean[:, 0].numpy(), std[:, 0].numpy()

plt.figure(figsize=(6, 3.8))
plt.plot(x_np, line_batch.fields[:, 0].numpy(), "k--", label="exact")
plt.plot(x_np, mean_np, label="BayesianNN mean")
plt.fill_between(x_np, mean_np - 2 * std_np, mean_np + 2 * std_np, alpha=0.3, label="mean +/- 2 std")
plt.xlabel("x")
plt.ylabel("u(x, 0.5)")
plt.legend()
plt.title("MC Dropout prediction at t = 0.5")
plt.tight_layout()
plt.show()

### 6.2 `EnsembleModel`

An ensemble wraps several models (of any mix of types) in an `nn.ModuleList`, so `parameters()` covers all members. The prediction is their `"mean"` or `"weighted_mean"`, and the **uncertainty reflects disagreement between members**: identical members give zero uncertainty, different members give positive uncertainty.

We train three networks from different random seeds and combine them.

In [ ]:
members = []
for seed in range(3):
    torch.manual_seed(seed)
    member = BasePINN(input_dim=2, output_dim=1, hidden_layers=3, hidden_size=32)
    member_config = TrainerConfig(max_epochs=150, val_frequency=10, log_frequency=150)
    PINNTrainer(model=member, problem=problem, solver=solver, config=member_config).fit(
        train_batches=train_batches, val_batches=val_batches
    )
    members.append(member)

ensemble = EnsembleModel(models=members, aggregation="mean")
print(ensemble.summary())

with torch.no_grad():
    ens_out = ensemble.forward(line_batch)

error = (ens_out.pred - line_batch.fields).abs()[:, 0].numpy()
unc = ens_out.uncertainty[:, 0].numpy()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(x_np, line_batch.fields[:, 0].numpy(), "k--", label="exact")
ax[0].plot(x_np, ens_out.pred[:, 0].numpy(), label="ensemble mean")
ax[0].set_title("Ensemble prediction at t = 0.5")
ax[0].legend()
ax[1].plot(x_np, error, label="absolute error")
ax[1].plot(x_np, unc, label="ensemble uncertainty")
ax[1].set_title("Error versus uncertainty")
ax[1].legend()
for a in ax:
    a.set_xlabel("x")
plt.tight_layout()
plt.show()

**Weighted aggregation, adding members, mixed types.** Weights are normalised to sum to one. `add_model()` grows the ensemble in place, and members do not have to share a class.

In [ ]:
weighted = EnsembleModel(models=members, aggregation="weighted_mean", weights=[0.5, 0.3, 0.2])
print("weights:", weighted.weights.tolist(), "| sum =", round(weighted.weights.sum().item(), 6))

# Extreme weights: the ensemble collapses onto the dominant member
dominant = EnsembleModel(models=members[:2], aggregation="weighted_mean", weights=[0.9999, 0.0001])
with torch.no_grad():
    close = torch.allclose(dominant.forward(line_batch).pred, members[0].forward(line_batch).pred, atol=1e-3)
print("weight 0.9999 on the first member reproduces it:", close)

# Grow the ensemble
ensemble.add_model(BasePINN(input_dim=2, output_dim=1, hidden_layers=3, hidden_size=32), weight=1.0)
print("members after add_model:", len(ensemble.models))

# Mixed model types
mixed = EnsembleModel(
    models=[
        BasePINN(input_dim=2, output_dim=1, hidden_layers=2, hidden_size=16),
        ResidualPINN(input_dim=2, output_dim=1, hidden_layers=2, hidden_size=16),
        BayesianNN(input_dim=2, output_dim=1, hidden_size=16, n_samples=5),
    ],
    aggregation="mean",
)
print("mixed ensemble pred shape:", tuple(mixed.forward(make_batch()).pred.shape))

# Identical members have nothing to disagree about
same = EnsembleModel(models=[members[0], members[0], members[0]])
with torch.no_grad():
    print("identical members -> mean uncertainty:", same.forward(line_batch).uncertainty.mean().item())

# Invalid configurations are rejected
for bad in (dict(models=[]), dict(models=members, aggregation="vote"), dict(models=members[:2], weights=[0.5, 0.3, 0.2])):
    try:
        EnsembleModel(**bad)
    except ValueError:
        print(f"EnsembleModel({list(bad)}) -> ValueError")

### 6.3 `UncertaintyMetrics`

Static helpers for judging uncertainty estimates. MC samples have shape `(n_samples, n_points, output_dim)`.

* `epistemic(samples)`: uncertainty from the model (spread across samples)
* `aleatoric(samples, targets)`: uncertainty attributed to the data
* `calibration_error(pred_mean, pred_std, targets)`: a float in `[0, 1]`; **lower means better calibrated**

In [ ]:
samples = torch.randn(20, 50, 1)      # 20 stochastic passes, 50 points, 1 output
targets = torch.zeros(50, 1)

print("epistemic shape:", tuple(UncertaintyMetrics.epistemic(samples).shape))
print("aleatoric shape:", tuple(UncertaintyMetrics.aleatoric(samples, targets).shape))

# Calibration of the trained BayesianNN along the t = 0.5 line
ece = UncertaintyMetrics.calibration_error(mean, std.clamp(min=1e-8), line_batch.fields)
print(f"calibration error of bayes_model on the t = 0.5 line: {ece:.3f}")

## 7. End to end

Everything together: two synthetic sources of different resolution become fidelity-ranked `FeatureBatch` objects, a multi-fidelity PINN learns from both, and validation uses a third source that went through the same transformer.

```
SyntheticDataSource x2 -> PhysicalTensor -> DataTransformer -> [FeatureBatch, FeatureBatch]
                                                                      |
                MultiFidelityPINN + HeatProblem + AutogradPDESolver -> MultiFidelityTrainer.fit()
```

In [ ]:
train_sources = [
    SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=100),
    SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=400),
]
val_source = SyntheticDataSource(solution_fn=heat_solution, config=make_config(), n_points=200)

e2e_train = DataTransformer().transform([s.load() for s in train_sources])
e2e_val = DataTransformer().transform([val_source.load()])
print("training fidelity levels:", [b.fidelity_level for b in e2e_train])
print("fidelity weights        :", [round(b.fidelity_weight, 3) for b in e2e_train])

e2e_model = MultiFidelityPINN(input_dim=2, output_dim=1, n_fidelity_levels=2, hidden_layers=3, hidden_size=32)
e2e_trainer = MultiFidelityTrainer(
    model=e2e_model,
    problem=HeatProblem(),
    solver=AutogradPDESolver(),
    config=TrainerConfig(max_epochs=200, val_frequency=10, log_frequency=50),
    mf_config=MultiFidelityConfig(curriculum=False, n_fidelity_levels=2),
)
e2e_history = e2e_trainer.fit(train_batches=e2e_train, val_batches=e2e_val)

plt.figure(figsize=(5, 3.5))
plt.semilogy(e2e_history.train_loss)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("End-to-end multi-fidelity training")
plt.tight_layout()
plt.show()

## 8. Limitations in v1

Some components exist as interfaces only, so that the architecture is complete and later versions can fill them in without breaking user code:

| Component | State in v1 |
|-----------|-------------|
| `HDF5DataSource` | Constructible; `validate()` works; loading is not implemented (waiting for real GENE / GS2 / CGYRO datasets) |
| `HasegawaWakatani.residual()` | Raises `NotImplementedError` (needs the Poisson bracket from `PlasmaPhysicsProblem`) |
| `RadiationProblem`, `NuclearMaterialProblem` | Stubs; `pde_residual()` raises `NotImplementedError` |
| `BayesianTrainer` | Standard loss without an explicit KL term (ADR-011) |

Planned for later versions: a command-line interface, reinforcement-learning training, full HDF5 loading, the remaining domain problems, and (in v3) multi-GPU training, model export, hyper-parameter sweeps and uncertainty-guided active learning.

The `tests/` folder is the best place to see each component exercised in detail; the sections of this notebook follow the same order as the test files.